# Track 1: Reasoned Financial Sentiment Fine-Tuning
This notebook demonstrates fine-tuning the **Gemma 4 E2B-IT** model on the `lmassaron/FinancialPhraseBank_explained` dataset using **TRL** and **LoRA** (4-bit QLoRA).

### Objectives
- Learn how to inject Chain-of-Thought (CoT) reasoning into a classification task.
- Perform Supervised Fine-Tuning (SFT) using Hugging Face `trl` and `peft` libraries.
- Optimize training parameters to fit within a 16GB VRAM constraint.

In [1]:
# Install extra dependencies if needed (e.g. on Google Colab)
# %pip install -U transformers trl peft accelerate bitsandbytes datasets

In [2]:
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Check device and capability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Using device: {device} | Dtype: {compute_dtype}')

W0720 00:13:48.379000 2450157 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0720 00:13:48.396000 2450157 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Using device: cuda | Dtype: torch.bfloat16


## 1. Load Dataset
We pull the `lmassaron/FinancialPhraseBank_explained` dataset which contains news sentences, ground-truth sentiments, and human-like explanations.

In [3]:
DATASET_ID = 'lmassaron/FinancialPhraseBank_explained'
print(f'Loading {DATASET_ID}...')
dataset = load_dataset(DATASET_ID)

# Rename columns to standard names
dataset = dataset.rename_columns({
    'sentence': 'text',
    'explanation': 'reasoning'
})
train_ds = dataset['train']
eval_ds = dataset['validation']
print(train_ds)
print('Sample sentence:', train_ds[0]['text'])
print('Sample sentiment:', train_ds[0]['sentiment'])
print('Sample explanation:', train_ds[0]['reasoning'])

Loading lmassaron/FinancialPhraseBank_explained...


Dataset({
    features: ['text', 'label', 'sentiment', 'reasoning', 'is_augmented', 'judge_score'],
    num_rows: 7902
})
Sample sentence: The shares carry a right to dividend and other shareholder rights as from their registration with the Finnish Trade Register .
Sample sentiment: neutral
Sample explanation: The inclusion of a right to dividend and other shareholder rights indicates that the shares have reached a status where they are eligible for dividends and certain voting rights. This typically does not directly impact revenue or profitability but can enhance investor confidence by providing clear entitlements and potentially increasing the perceived value of the shares, thus maintaining a neutral sentiment in terms of market reaction.


## 2. Formatting Prompts using Chat Template
We structure the news headline, instructions, and target sentiment/explanations into a chat template using Gemma 4's format.

In [4]:
SYSTEM_PROMPT = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)

MODEL_ID = 'google/gemma-4-E2B-it'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example):
    messages = [
        {'role': 'user', 'content': SYSTEM_PROMPT + f'\nHeadline: "{example["text"]}"'},
        {
            'role': 'assistant',
            'content': f'<sentiment>{example["sentiment"]}</sentiment>\n<reasoning>{example["reasoning"]}</reasoning>'
        }
    ]
    # Format without tokenizing so SFTTrainer can tokenize it
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': prompt}

train_mapped = train_ds.map(format_prompt)
eval_mapped = eval_ds.map(format_prompt)
print('Mapped Prompt Preview:\n', train_mapped[0]['text'])

Mapped Prompt Preview:
 <bos><|turn>user
You are a financial analyst with expertise in equity markets and corporate finance.
Analyze the following financial news headline and determine its market sentiment from an investor's perspective.

Classify the sentiment as positive, neutral, or negative based on the likely impact on stock price, investor confidence, or financial performance.

Respond using exactly these two tags:
<sentiment>positive|neutral|negative</sentiment>
<reasoning>brief financial explanation</reasoning>

Headline: "The shares carry a right to dividend and other shareholder rights as from their registration with the Finnish Trade Register ."<turn|>
<|turn>model
<sentiment>neutral</sentiment>
<reasoning>The inclusion of a right to dividend and other shareholder rights indicates that the shares have reached a status where they are eligible for dividends and certain voting rights. This typically does not directly impact revenue or profitability but can enhance investor conf

## 3. Load Model with 4-bit Quantization
We load the base `google/gemma-4-E2B-it` model in 4-bit precision to fit within the 16GB VRAM constraint, and configure LoRA adapters targeting all linear modules.

In [5]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto'
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## 4. Run SFT Trainer
We configure the Hugging Face `trl` SFT Config and start fine-tuning. This is optimized for a batch size of 2 and gradient accumulation of 4 (effective batch size of 8).

In [6]:
training_args = SFTConfig(
    output_dir='gemma4-sentiment-lora',
    dataset_text_field='text',
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim='adamw_torch_fused',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=20,
    report_to='none',
    dataset_kwargs={
        'add_special_tokens': False,
    },
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# Disable KV cache during training to save VRAM
model.config.use_cache = False

trainer.train()

# Save the fine-tuned adapter
trainer.model.save_pretrained('gemma4-sentiment-lora-adapter')
tokenizer.save_pretrained('gemma4-sentiment-lora-adapter')
print('Adapter successfully saved!')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/7902 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7902 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7902 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/988 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/988 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/988 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,3.266977,2.386313,0.895824,34587.000000,0.589561
40,1.042999,0.904189,0.874418,69625.000000,0.803470
60,0.794317,0.788712,0.764530,104400.000000,0.820044
80,0.777839,0.761969,0.781921,139195.000000,0.824194
100,0.722752,0.741675,0.725368,173598.000000,0.826338
120,0.696425,0.720676,0.710434,208191.000000,0.831439
140,0.691209,0.709446,0.712042,242606.000000,0.833015
160,0.707619,0.700196,0.713093,277456.000000,0.834411
180,0.677467,0.693346,0.638896,312214.000000,0.836580
200,0.666432,0.688544,0.673648,346499.000000,0.836833


Adapter successfully saved!


## 5. Evaluation and Inference
We load the fine-tuned adapter, merge it with the base model, and run batch evaluation on the original (non-augmented) test set Headlines. We measure accuracy, tag format integrity, and output a detailed classification report.

In [7]:
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import re
import pandas as pd

# 1. Load fine-tuned model
print('Loading merged base model and adapter...')
eval_model = PeftModel.from_pretrained(model, 'gemma4-sentiment-lora-adapter')
eval_model = eval_model.eval()

# 2. Prepare test data (filtering out augmented samples for clean evaluation)
test_df = dataset['test'].to_pandas()
if 'is_augmented' in test_df.columns:
    test_df = test_df[test_df['is_augmented'] == False].reset_index(drop=True)
print(f'Evaluating on {len(test_df)} original test headlines.')

# 3. Configure tokenizer for batch left-padding
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def extract_tag(text, tag):
    match = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

results = []
batch_size = 16

for start in tqdm(range(0, len(test_df), batch_size), desc='Evaluating'):
    batch_df = test_df.iloc[start : start + batch_size]
    prompts = [
        tokenizer.apply_chat_template(
            [
                {'role': 'user', 'content': SYSTEM_PROMPT + f'Headline: "{row["text"]}"'}
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for _, row in batch_df.iterrows()
    ]
    
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    input_len = inputs['input_ids'].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    
    for i, raw_output in enumerate(decoded):
        sentiment_tag = extract_tag(raw_output, 'sentiment')
        reasoning_pred = extract_tag(raw_output, 'reasoning')
        results.append({
            'sentiment_true': batch_df.iloc[i]['sentiment'],
            'sentiment_pred': sentiment_tag.lower() if sentiment_tag else 'none',
            'reasoning_pred': reasoning_pred if reasoning_pred else 'none',
            'tag_integrity': sentiment_tag is not None and reasoning_pred is not None
        })

results_df = pd.DataFrame(results)
accuracy = accuracy_score(results_df['sentiment_true'], results_df['sentiment_pred'])
integrity = results_df['tag_integrity'].mean()
print(f'\nAccuracy: {accuracy:.2%}')
print(f'Tag Integrity: {integrity:.2%}')
print("\nClassification Report:")
print(classification_report(results_df['sentiment_true'], results_df['sentiment_pred']))

Loading merged base model and adapter...


Evaluating on 484 original test headlines.


Evaluating:   0%|                                | 0/31 [00:00<?, ?it/s]

Evaluating:   3%|▊                       | 1/31 [00:07<03:51,  7.72s/it]

Evaluating:   6%|█▌                      | 2/31 [00:14<03:34,  7.40s/it]

...

Evaluating:  94%|█████████████████████▌ | 29/31 [03:31<00:14,  7.39s/it]

Evaluating:  97%|██████████████████████▎| 30/31 [03:38<00:07,  7.36s/it]

Evaluating: 100%|███████████████████████| 31/31 [03:44<00:00,  7.06s/it]

Evaluating: 100%|███████████████████████| 31/31 [03:44<00:00,  7.25s/it]


Accuracy: 81.82%
Tag Integrity: 94.83%

Classification Report:
              precision    recall  f1-score   support

    negative       0.88      0.85      0.87        61
     neutral       0.84      0.89      0.87       287
        none       0.00      0.00      0.00         0
    positive       0.91      0.65      0.76       136

    accuracy                           0.82       484
   macro avg       0.66      0.60      0.62       484
weighted avg       0.87      0.82      0.84       484

